# Közel inkompresszibilis blokk — visszaszámított hidrosztatikus nyomás
Borsodi Bálint TDK-dolgozata, B5 (7.5. szakasz). 6×6 elemes egységnégyzet, alul befogva, a felső élre 1 MPa nyomás jut. A visszaszámítás az SRI elmozdulásmezőből történik: $p_e=-\kappa\,\overline{\mathrm{div}\,u}_e$.
Az affin négyszögeken a négy sarokérték átlaga az elemátlagot adja. A számított nyomás elemenkénti és simítatlan.

In [1]:
include(joinpath(@__DIR__, "q4_common.jl"))

A három nyomásstatisztika és a függőleges erőegyensúly ellenőrzése.

In [2]:
E, ν = 50e9, 0.4999
U, _ = q4_grid(6, 6)
bc = [BoundaryCondition("bottom"; field=U, ux=0, uy=0)]
f = ∫(U ⋅ [0.0, -1e6]; Γ="top")
_, K = q4_stiffness(U, E, ν)
u = solveField(K, f; support=bc)

κ = E / (3(1-2ν))
divu = ∇(expandTo3D(u); nabla=:div)
p = [-κ * mean(values) / 1e6 for values in divu.A] # MPa
stats = [minimum(p), maximum(p), mean(p)]
@assert length(p) == 36
@assert maximum(abs, stats - [0.392, 0.878, 0.564]) < 0.0006

R = K.A * u.a - f.a
fixed = constrainedDoFs(U, bc)
Ry = sum(R[filter(iseven, fixed),1])
Fy = sum(f.a[2:2:end,1])
@assert abs((Ry+Fy)/Fy) < 1e-9
@printf("p [MPa]: min %.3f, max %.3f, átlag %.3f\n", stats...)
@printf("Fᵧ = %.0f N, Rᵧ = %+.0f N\n", Fy, Ry)

p [MPa]: min 0.392, max 0.878, átlag 0.564
Fᵧ = -1000000 N, Rᵧ = +1000000 N
